# 🔄 Data Pipeline — Building a Complete Data Processing Workflow
**Step 2: Data Science → Topic 09 → Class 25** | Date: February 22, 2024 🎯

> **A Data Pipeline takes data from somewhere → processes it → cleans/transforms it → sends it somewhere useful.**

```text
Source → Ingest → Raw → Validate → Clean → Transform → Validate → Store → Analyze/ML → Monitor
```

## 🎯 Learning Objectives
Pipeline basics • sources & data types • ingestion • batch / stream / micro-batch • ETL vs ELT • full vs incremental load • CDC • warehouse / lake / lakehouse • scheduling, DAGs, orchestration • logging, error handling, retries • **idempotency** • data quality, lineage, schema validation • Bronze/Silver/Gold • quarantine data • ML & feature pipelines • RAG ingestion pipeline • testing • Docker • mini project

💡 **Everything below runs offline** — the notebook creates its own messy sample data under `pipeline_demo/`.

## 🧠 Mental Model
```text
SOURCE → INGEST → RAW → VALIDATE → CLEAN → TRANSFORM → VALIDATE → STORE → ANALYZE / ML / APP → MONITOR
```
**Pipeline ≠ script.** A script reads → cleans → saves. A production pipeline adds scheduling, dependencies, validation, quality checks, logging, monitoring, retries and alerts.

## 🛠️ 0. Setup

In [ ]:
import os, json, time, sqlite3, logging, itertools, random
from pathlib import Path
from datetime import datetime, timedelta
from functools import wraps

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

# Project folders (matches the course folder structure)
BASE = Path("pipeline_demo")
RAW, PROCESSED, OUTPUT = BASE / "data/raw", BASE / "data/processed", BASE / "output"
for p in (RAW, PROCESSED, OUTPUT):
    p.mkdir(parents=True, exist_ok=True)

# Logging (force=True so re-running the cell doesn't duplicate handlers)
import sys
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", stream=sys.stdout, force=True)
log = logging.getLogger("pipeline")
print("Setup complete ✅ →", BASE.resolve())

## 📥 1. Data Sources & Types of Data

| Source | Examples |
|---|---|
| Files | CSV, Excel, JSON, XML, Parquet |
| Databases | PostgreSQL, MySQL, MongoDB |
| APIs | REST, GraphQL, third-party |
| Logs / Events | Server logs, clicks, purchases, sensor events |

| Type | Example |
|---|---|
| **Structured** | SQL tables, CSV |
| **Semi-structured** | JSON, XML, API responses |
| **Unstructured** | PDF, images, audio, emails, text |

### 📦 Our sample data (deliberately messy!)
We create `customers.csv`, `products.csv`, `orders.csv` with real-world problems: extra spaces, `"1,200"` prices, duplicates, negative quantities, invalid dates, missing IDs, orphan customer IDs.

In [ ]:
rng = np.random.default_rng(42)

# --- customers (50 + 2 duplicate rows + messy names/emails) ---
names = ["Ali Khan", "Ahmed Raza", "Sara Malik", "Hamza Iqbal", "Ayesha Noor", "Bilal Shah", "Fatima Zahra", "Usman Ali"]
customers = pd.DataFrame({
    "customer_id": range(1, 51),
    "name": [f"  {rng.choice(names).upper() if i % 4 == 0 else rng.choice(names)}  " for i in range(50)],
    "email": [f"User{i}@Example.COM " if i % 3 == 0 else f"user{i}@example.com" for i in range(1, 51)],
})
customers.loc[[7, 19], "email"] = np.nan
customers = pd.concat([customers, customers.iloc[[3, 4]]], ignore_index=True)   # duplicates

# --- products (price stored as text, some with commas) ---
products = pd.DataFrame({
    "product_id": range(101, 111),
    "name": ["Laptop", "Phone", "Tablet", "Mouse", "Keyboard", "Monitor", "Headphones", "Webcam", "Charger", "Speaker"],
    "price": ["100,000", "60000", "45,000", "1500", "3,200", "28000", "5,500", "4000", "1200", "-50"],  # last is invalid
})

# --- orders (300 + duplicates + bad values) ---
n = 300
orders = pd.DataFrame({
    "order_id": range(1, n + 1),
    "customer_id": rng.integers(1, 52, n).astype(float),        # id 51 = orphan (not in customers)
    "product_id": rng.integers(101, 111, n),
    "quantity": rng.integers(1, 5, n),
    "date": (pd.Timestamp("2024-02-01") + pd.to_timedelta(rng.integers(0, 21, n), unit="D")).strftime("%Y-%m-%d"),
})
orders.loc[[10, 50], "quantity"] = [0, -2]            # invalid quantities
orders.loc[[20], "date"] = "not_a_date"               # invalid date
orders.loc[[30, 31], "customer_id"] = np.nan          # missing customer id
orders = pd.concat([orders, orders.iloc[[5, 6]]], ignore_index=True)   # duplicate order rows

customers.to_csv(RAW / "customers.csv", index=False)
products.to_csv(RAW / "products.csv", index=False)
orders.to_csv(RAW / "orders.csv", index=False)
print("Raw files created:", [f.name for f in RAW.glob("*.csv")])
orders.head()

## 📥 2. Extraction — CSV, JSON/API, SQL
Extraction = getting data from its source.

In [ ]:
# 1) CSV
orders_raw = pd.read_csv(RAW / "orders.csv")
print("CSV rows:", len(orders_raw))

# 2) JSON / API response (semi-structured) → flatten to a table
api_response = {
    "data": [
        {"name": "Ali", "age": 23, "skills": ["Python", "SQL"], "address": {"city": "Islamabad"}},
        {"name": "Ahmed", "age": 25, "skills": ["React"], "address": {"city": "Lahore"}},
    ]
}
pd.json_normalize(api_response["data"])           # nested dict → columns like address.city

In [ ]:
# 3) Real API call pattern (needs internet — shown as reference)
# import requests
# response = requests.get("https://api.example.com/orders", timeout=10)
# response.raise_for_status()          # raise on 4xx/5xx
# data = response.json()

# 4) SQL source (using SQLite as a stand-in for PostgreSQL/MySQL)
conn = sqlite3.connect(BASE / "source.db")
orders_raw.to_sql("orders", conn, if_exists="replace", index=False)
pd.read_sql("SELECT customer_id, COUNT(*) AS n FROM orders GROUP BY customer_id ORDER BY n DESC LIMIT 5", conn)

## 🔄 3. Batch vs Streaming vs Micro-Batch

| Batch | Streaming |
|---|---|
| Groups at scheduled times | Continuous, event-by-event |
| Higher latency | Low latency |
| Easier | More complex |
| Daily reports, ML training sets | Fraud detection, live analytics, IoT |

**Micro-batch** = process small batches frequently (e.g. every 10 seconds).

In [ ]:
# ⚡ Simulate a stream of click events with a generator
def event_stream(n_events=12):
    for i in range(n_events):
        yield {"event_id": i, "user": random.randint(1, 5), "action": random.choice(["click", "view", "buy"])}

# Streaming: handle each event immediately
print("— STREAMING —")
for event in itertools.islice(event_stream(4), 4):
    print("processed:", event)

# Micro-batch: collect N events, process together
def micro_batches(stream, size):
    while True:
        batch = list(itertools.islice(stream, size))
        if not batch:
            return
        yield batch

print("\n— MICRO-BATCH (size=4) —")
for i, batch in enumerate(micro_batches(event_stream(12), 4), 1):
    print(f"batch {i}: {len(batch)} events | buys = {sum(e['action'] == 'buy' for e in batch)}")

## 🧹 4. Clean → Transform → Validate (the core stages)

- **Clean:** missing values, duplicates, wrong types, extra spaces, inconsistent categories
- **Transform:** reshape into what downstream needs (`total = price × quantity`, date parts, joins)
- **Validate:** business rules (price ≥ 0, quantity > 0, unique IDs, valid emails)

> ⚠️ **Pipeline success ≠ data quality success.** A pipeline can run perfectly and still produce wrong data.

In [ ]:
# ---------- EXTRACT ----------
def extract_data(raw_dir: Path = RAW) -> dict:
    """Read all raw CSVs. Raw files are never modified (keep raw data!)."""
    data = {name: pd.read_csv(raw_dir / f"{name}.csv") for name in ("customers", "products", "orders")}
    log.info("Extracted %s", {k: len(v) for k, v in data.items()})
    return data

# ---------- CLEAN ----------
def clean_data(data: dict) -> dict:
    c, p, o = (data[k].copy() for k in ("customers", "products", "orders"))

    # customers: trim, title-case names, lower-case emails, drop duplicate IDs
    c["name"] = c["name"].str.strip().str.title()
    c["email"] = c["email"].str.strip().str.lower()
    c = c.drop_duplicates(subset="customer_id", keep="first")

    # products: "100,000" → 100000 (invalid → NaN)
    p["price"] = pd.to_numeric(p["price"].astype(str).str.replace(",", "", regex=False), errors="coerce")

    # orders: dates → datetime (invalid → NaT), drop exact duplicate order_ids
    o["date"] = pd.to_datetime(o["date"], errors="coerce")
    o = o.drop_duplicates(subset="order_id", keep="first")

    log.info("Cleaned → customers=%d products=%d orders=%d", len(c), len(p), len(o))
    return {"customers": c, "products": p, "orders": o}

clean = clean_data(extract_data())
clean["orders"].dtypes

## ✅ 5. Data Validation + 🚨 Quarantine (Dead-Letter) Records
Bad records shouldn't kill the whole pipeline — send them to a **rejected** file with a reason, and keep going.

In [ ]:
def validate_orders(orders: pd.DataFrame, customers: pd.DataFrame, products: pd.DataFrame):
    """Return (valid, rejected). Every rejected row carries a reason."""
    o = orders.copy()
    valid_prices = products.loc[products["price"] >= 0, "product_id"]

    checks = {
        "missing_customer_id": o["customer_id"].isna(),
        "unknown_customer":    ~o["customer_id"].isin(customers["customer_id"]) & o["customer_id"].notna(),
        "unknown_or_bad_product": ~o["product_id"].isin(valid_prices),
        "invalid_quantity":    o["quantity"] <= 0,
        "invalid_date":        o["date"].isna(),
    }
    o["reject_reason"] = ""
    for reason, mask in checks.items():
        o.loc[mask & (o["reject_reason"] == ""), "reject_reason"] = reason

    rejected = o[o["reject_reason"] != ""]
    valid = o[o["reject_reason"] == ""].drop(columns="reject_reason")
    log.info("Validation → valid=%d rejected=%d", len(valid), len(rejected))
    return valid, rejected

valid_orders, rejected_orders = validate_orders(clean["orders"], clean["customers"], clean["products"])
rejected_orders.to_csv(OUTPUT / "rejected_orders.csv", index=False)       # quarantine file
rejected_orders["reject_reason"].value_counts()

In [ ]:
# Data-quality assertions (fail loudly if something slips through)
assert valid_orders["order_id"].is_unique, "duplicate order IDs!"
assert valid_orders["customer_id"].notna().all(), "missing customer IDs!"
assert (valid_orders["quantity"] > 0).all(), "non-positive quantity!"
print("All quality assertions passed ✅")

## 🔄 6. Transform + Aggregate
Join orders + customers + products, compute `total_amount`, then build summary tables.

In [ ]:
def transform_data(orders, customers, products) -> pd.DataFrame:
    df = (orders
          .merge(customers[["customer_id", "name", "email"]], on="customer_id", how="left")
          .merge(products.rename(columns={"name": "product_name"}), on="product_id", how="left"))
    df["customer_id"] = df["customer_id"].astype(int)
    df["total_amount"] = df["price"] * df["quantity"]                 # transformation
    df["year"], df["month"], df["day"] = df["date"].dt.year, df["date"].dt.month, df["date"].dt.day
    return df

def aggregate_data(df: pd.DataFrame) -> dict:
    return {
        "daily_sales": df.groupby(df["date"].dt.date).agg(orders=("order_id", "count"), revenue=("total_amount", "sum")).reset_index(),
        "customer_summary": df.groupby(["customer_id", "name"]).agg(
            total_orders=("order_id", "count"), total_spending=("total_amount", "sum"),
            avg_order_value=("total_amount", "mean")).reset_index(),
        "product_summary": df.groupby("product_name").agg(units=("quantity", "sum"), revenue=("total_amount", "sum")).reset_index()
                             .sort_values("revenue", ascending=False),
    }

processed_orders = transform_data(valid_orders, clean["customers"], clean["products"])
summaries = aggregate_data(processed_orders)
print("Average order value:", round(processed_orders["total_amount"].mean(), 2))
summaries["daily_sales"].head()

## 🗄️ 7. Storage — Warehouse vs Lake vs Lakehouse

| Data Lake | Data Warehouse |
|---|---|
| Raw, many formats (CSV, JSON, Parquet, images) | Curated, structured |
| Data Science / ML | BI / analytics / SQL |
| Flexible | Controlled schema |

**Lakehouse** = lake storage + warehouse-like management (SQL + ML on one platform).
Tools: **Warehouse** → Snowflake, BigQuery, Redshift • **Lake storage** → AWS S3, Azure Blob, GCS

In [ ]:
# "Lake" style: files (CSV / Parquet)   |   "Warehouse" style: SQL tables (SQLite stand-in)
processed_orders.to_csv(PROCESSED / "processed_orders.csv", index=False)
for name, table in summaries.items():
    table.to_csv(PROCESSED / f"{name}.csv", index=False)

try:
    processed_orders.to_parquet(PROCESSED / "processed_orders.parquet", index=False)   # columnar, compressed
    print("Parquet saved ✅")
except Exception as e:
    print("Parquet skipped (pip install pyarrow):", type(e).__name__)

wh = sqlite3.connect(BASE / "warehouse.db")
processed_orders.assign(date=processed_orders["date"].astype(str)).to_sql("fact_orders", wh, if_exists="replace", index=False)
print(pd.read_sql("SELECT COUNT(*) AS rows_in_warehouse FROM fact_orders", wh))

## 🔄 8. ETL vs ELT

```text
ETL:  Source → Transform → Destination     (transform BEFORE loading)
ELT:  Source → Destination → Transform     (load raw first, transform IN the destination)
```
ELT is common on modern cloud platforms whose warehouse can transform efficiently (dbt-style SQL).

In [ ]:
# ETL — transform in Python, then load
etl_result = transform_data(valid_orders, clean["customers"], clean["products"])

# ELT — load RAW first, then transform inside the database using SQL
elt = sqlite3.connect(":memory:")
clean["orders"].assign(date=clean["orders"]["date"].astype(str)).to_sql("raw_orders", elt, index=False)
clean["products"].to_sql("raw_products", elt, index=False)

elt.executescript("""
CREATE TABLE gold_product_sales AS
SELECT p.name AS product, SUM(o.quantity) AS units, SUM(o.quantity * p.price) AS revenue
FROM raw_orders o JOIN raw_products p ON o.product_id = p.product_id
WHERE o.quantity > 0 AND p.price >= 0
GROUP BY p.name;
""")
pd.read_sql("SELECT * FROM gold_product_sales ORDER BY revenue DESC LIMIT 5", elt)

## ➕ 9. Full Load vs Incremental Load + 🔄 CDC

| Full | Incremental |
|---|---|
| Process **everything** every time | Process only **new/changed** data |
| Simple, but expensive at scale | Cheaper and faster |

**CDC (Change Data Capture)** = detect `INSERT` / `UPDATE` / `DELETE` in a source system and forward only those changes.

In [ ]:
# Incremental load using a WATERMARK (last processed date), stored in a state file
STATE = BASE / "state.json"
STATE.unlink(missing_ok=True)

def load_watermark():
    return pd.Timestamp(json.loads(STATE.read_text())["watermark"]) if STATE.exists() else pd.Timestamp.min

def incremental_batch(df: pd.DataFrame, up_to: pd.Timestamp):
    """Simulate 'today's run': only rows newer than the watermark and up to `up_to`."""
    wm = load_watermark()
    new = df[(df["date"] > wm) & (df["date"] <= up_to)]
    if not new.empty:
        STATE.write_text(json.dumps({"watermark": str(new["date"].max())}))
    return new

full = processed_orders
run1 = incremental_batch(full, pd.Timestamp("2024-02-10"))
run2 = incremental_batch(full, pd.Timestamp("2024-02-21"))
run3 = incremental_batch(full, pd.Timestamp("2024-02-21"))    # nothing new
print(f"Full dataset: {len(full)} | Run1: {len(run1)} | Run2: {len(run2)} | Run3 (re-run): {len(run3)}")
print("Run1 + Run2 = ", len(run1) + len(run2), "(no double-processing ✅)")

In [ ]:
# CDC concept: compare two snapshots by key → INSERT / UPDATE / DELETE
old = pd.DataFrame({"id": [1, 2, 3], "email": ["a@x.com", "b@x.com", "c@x.com"]})
new = pd.DataFrame({"id": [2, 3, 4], "email": ["b@x.com", "c_new@x.com", "d@x.com"]})

m = old.merge(new, on="id", how="outer", suffixes=("_old", "_new"), indicator=True)
inserts = m[m["_merge"] == "right_only"]
deletes = m[m["_merge"] == "left_only"]
updates = m[(m["_merge"] == "both") & (m["email_old"] != m["email_new"])]
print(f"INSERT: {inserts['id'].tolist()} | UPDATE: {updates['id'].tolist()} | DELETE: {deletes['id'].tolist()}")
# 🧠 Real CDC reads the database transaction log (e.g. Debezium) instead of comparing full snapshots.

## 🚨 10. Logging, Error Handling & 🔁 Retry

Logs answer "what happened?". Errors will happen (API down, bad data, schema change, network failure).
Retry **temporary** failures — but with a **limit** and **backoff**, never infinite.

In [ ]:
def retry(max_attempts=3, delay=0.1, backoff=2.0, exceptions=(ConnectionError, TimeoutError)):
    """Retry decorator with exponential backoff. Only retries the exceptions you list."""
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            wait = delay
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as err:
                    log.warning("%s failed (attempt %d/%d): %s", fn.__name__, attempt, max_attempts, err)
                    if attempt == max_attempts:
                        log.error("Giving up after %d attempts", max_attempts)
                        raise
                    time.sleep(wait); wait *= backoff
        return wrapper
    return decorator

calls = {"n": 0}

@retry(max_attempts=4)
def flaky_api():
    calls["n"] += 1
    if calls["n"] < 3:                         # fails twice, succeeds on the 3rd try
        raise ConnectionError("API unavailable")
    return {"status": "ok", "attempt": calls["n"]}

print(flaky_api())

In [ ]:
# try / except / finally in a pipeline step — fail loudly on real bugs, handle known failure modes
def safe_extract(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        log.error("Source file missing: %s", path)
        raise
    except pd.errors.EmptyDataError:
        log.error("Source file is empty: %s", path)
        return pd.DataFrame()
    finally:
        log.info("extract step finished for %s", Path(path).name)

try:
    safe_extract(RAW / "does_not_exist.csv")
except FileNotFoundError:
    print("→ pipeline stopped cleanly with a clear error ✅")

## 🆔 11. Idempotency ⭐ (Very Common Interview Topic)

> Running the same operation multiple times gives the **same final result** — no duplicates, no corruption.

**Problem:** the pipeline fails halfway → you re-run → the same 1,000 rows are inserted again.
**Solutions:** unique IDs • **UPSERT / MERGE** • replace partitions • transactions • deduplication

In [ ]:
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, total REAL)")
rows = [(1, 10, 500.0), (2, 11, 900.0), (3, 12, 150.0)]

# ❌ NOT idempotent: plain INSERT, run twice → duplicates / crash
def load_bad(rows):
    try:
        db.executemany("INSERT INTO orders VALUES (?, ?, ?)", rows)
    except sqlite3.IntegrityError as e:
        print("  ❌ second run fails:", e)

# ✅ Idempotent: UPSERT
def load_upsert(rows):
    db.executemany("""
        INSERT INTO orders (order_id, customer_id, total) VALUES (?, ?, ?)
        ON CONFLICT(order_id) DO UPDATE SET customer_id = excluded.customer_id, total = excluded.total
    """, rows)
    db.commit()

load_bad(rows); load_bad(rows)
db.execute("DELETE FROM orders")
load_upsert(rows); load_upsert(rows); load_upsert(rows)         # run 3 times
print("✅ Rows after 3 upsert runs:", db.execute("SELECT COUNT(*) FROM orders").fetchone()[0], "(still 3)")

## 📊 12. Data Quality & Schema Validation

**Dimensions:** Accuracy • Completeness • Consistency • Validity • Uniqueness • Timeliness

**Schema change risk:** API changes `customer_id → id` and your pipeline breaks → validate the schema *before* processing.

In [ ]:
EXPECTED_SCHEMA = {"order_id": "int", "customer_id": "float", "product_id": "int", "quantity": "int", "date": "object"}

def validate_schema(df: pd.DataFrame, expected: dict) -> None:
    missing = set(expected) - set(df.columns)
    extra = set(df.columns) - set(expected)
    if missing:
        raise ValueError(f"Schema validation failed — missing columns: {sorted(missing)}")
    if extra:
        log.warning("Unexpected new columns (schema drift?): %s", sorted(extra))

validate_schema(orders_raw, EXPECTED_SCHEMA)
print("Schema OK ✅")

try:
    validate_schema(orders_raw.rename(columns={"customer_id": "id"}), EXPECTED_SCHEMA)
except ValueError as e:
    print("Caught schema change →", e)

In [ ]:
def data_quality_report(df: pd.DataFrame, key: str) -> pd.Series:
    return pd.Series({
        "rows": len(df),
        "completeness_%": round(100 * (1 - df.isna().mean().mean()), 2),          # non-null share
        "uniqueness_%": round(100 * df[key].nunique() / len(df), 2),              # key uniqueness
        "duplicates": int(df.duplicated(subset=key).sum()),
        "freshness_max_date": str(pd.to_datetime(df["date"], errors="coerce").max().date()),
    })

print("RAW ORDERS      →\n", data_quality_report(orders_raw, "order_id"), "\n")
print("PROCESSED ORDERS→\n", data_quality_report(processed_orders, "order_id"))

## 🧱 13. Bronze / Silver / Gold (Medallion) + 🧬 Lineage

```text
Bronze  = raw data, as received          (Raw Orders)
Silver  = cleaned & standardized         (Clean Orders)
Gold    = business-ready aggregates      (Daily Sales Summary)
```
**Lineage** answers: *"Where did this dashboard number come from?"*

In [ ]:
LINEAGE = []
def record(step, source, target, rows):
    LINEAGE.append({"time": datetime.now().strftime("%H:%M:%S"), "step": step, "source": source, "target": target, "rows": rows})

bronze = orders_raw
record("ingest", "orders.csv", "bronze_orders", len(bronze))

silver = valid_orders
record("clean+validate", "bronze_orders", "silver_orders", len(silver))

gold = summaries["daily_sales"]
record("aggregate", "silver_orders", "gold_daily_sales", len(gold))

pd.DataFrame(LINEAGE)

## 🕒 14. Scheduling, 🔗 Dependencies, 🕸️ DAG & ⚙️ Orchestration

Some tasks must run before others. A **DAG** (Directed Acyclic Graph) encodes those dependencies — directed (has direction), acyclic (no circular loops).

```text
        Extract
        /     \
   Customers  Orders
        \     /
         Join
          ↓
       Validate
          ↓
         Load
```
**Orchestrators** (Airflow, Prefect, Dagster) manage scheduling, dependencies, retries, failures, logging, monitoring, notifications.

In [ ]:
from graphlib import TopologicalSorter        # built into Python 3.9+

# task → set of tasks it depends on
dag = {
    "extract_customers": set(),
    "extract_orders": set(),
    "clean_customers": {"extract_customers"},
    "clean_orders": {"extract_orders"},
    "join": {"clean_customers", "clean_orders"},
    "validate": {"join"},
    "load": {"validate"},
}
order = list(TopologicalSorter(dag).static_order())
print("Execution order:", " → ".join(order))

# Mini-orchestrator: run tasks in dependency order and stop downstream tasks if one fails
def run_dag(dag, tasks):
    status = {}
    for name in TopologicalSorter(dag).static_order():
        if any(status.get(dep) != "success" for dep in dag[name]):
            status[name] = "skipped"; log.warning("SKIP %s (upstream failed)", name); continue
        try:
            tasks[name](); status[name] = "success"; log.info("OK   %s", name)
        except Exception as e:
            status[name] = "failed"; log.error("FAIL %s: %s", name, e)
    return status

tasks = {name: (lambda: None) for name in dag}
def boom(): raise RuntimeError("schema mismatch")
tasks["clean_orders"] = boom                              # simulate a failure
print(run_dag(dag, tasks))

In [ ]:
# ⏰ Scheduling simulation: what would run when?
def schedule_preview(start, every_minutes, runs):
    return [(start + timedelta(minutes=every_minutes * i)).strftime("%a %H:%M") for i in range(runs)]

print("Every 30 min :", schedule_preview(datetime(2024, 2, 22, 1, 0), 30, 5))
print("Cron 0 1 * * * = every day at 01:00 | */5 * * * * = every 5 minutes | 0 0 * * 1 = every Monday 00:00")

### 📄 Airflow DAG (reference — you'll use this later in MLOps)
```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

default_args = {"retries": 3, "retry_delay": timedelta(minutes=5)}

with DAG("ecommerce_daily", start_date=datetime(2024, 2, 22),
         schedule="0 1 * * *", catchup=False, default_args=default_args) as dag:
    extract   = PythonOperator(task_id="extract",   python_callable=extract_data)
    clean     = PythonOperator(task_id="clean",     python_callable=clean_data)
    transform = PythonOperator(task_id="transform", python_callable=transform_data)
    validate  = PythonOperator(task_id="validate",  python_callable=validate_data)
    load      = PythonOperator(task_id="load",      python_callable=load_data)

    extract >> clean >> transform >> validate >> load     # dependency chain
```

## ⚙️ 15. Pipeline Monitoring & 🔐 Security
Monitor: success/failure, duration, records processed/rejected, freshness, quality.
Security: authentication, encryption, secrets management, access control, audit logs, data masking.

In [ ]:
# 🔐 Never hard-code credentials. Read from environment variables (or a secrets manager).
DB_PASSWORD = os.getenv("DB_PASSWORD")            # export DB_PASSWORD=... in your shell / .env / CI secret
print("DB_PASSWORD set?", DB_PASSWORD is not None)

# 🎭 Data masking: hide PII before it lands in analytics tables
def mask_email(email: str) -> str:
    if not isinstance(email, str) or "@" not in email:
        return email
    user, domain = email.split("@", 1)
    return user[:2] + "***@" + domain

print(mask_email("waleed.khokhar@example.com"))

## 🧩 16. The Complete Pipeline — Everything Together
Extract → Clean → Validate → Transform → Aggregate → Load (idempotent) with logging, metrics, quarantine and error handling.

In [ ]:
class DataPipeline:
    def __init__(self, raw_dir=RAW, out_dir=PROCESSED, db_path=BASE / "warehouse.db"):
        self.raw_dir, self.out_dir, self.db_path = raw_dir, out_dir, db_path
        self.metrics = {}

    def run(self) -> dict:
        start = time.time()
        self.metrics = {"status": "RUNNING", "started": datetime.now().isoformat(timespec="seconds")}
        try:
            data = extract_data(self.raw_dir)
            self.metrics["input_records"] = len(data["orders"])

            validate_schema(data["orders"], EXPECTED_SCHEMA)               # fail fast on schema drift
            clean = clean_data(data)
            valid, rejected = validate_orders(clean["orders"], clean["customers"], clean["products"])
            rejected.to_csv(OUTPUT / "rejected_orders.csv", index=False)   # quarantine

            df = transform_data(valid, clean["customers"], clean["products"])
            assert df["order_id"].is_unique and (df["total_amount"] >= 0).all(), "output validation failed"

            summaries = aggregate_data(df)
            self._load(df, summaries)

            self.metrics.update(status="SUCCESS", processed_records=len(df), rejected_records=len(rejected))
        except Exception as err:
            self.metrics.update(status="FAILED", error=str(err))
            log.exception("Pipeline failed")            # full traceback in logs
            raise
        finally:
            self.metrics["duration_sec"] = round(time.time() - start, 2)
            log.info("Metrics: %s", self.metrics)
        return self.metrics

    def _load(self, df, summaries):
        df.to_csv(self.out_dir / "processed_orders.csv", index=False)
        for name, table in summaries.items():
            table.to_csv(self.out_dir / f"{name}.csv", index=False)
        with sqlite3.connect(self.db_path) as con:                          # idempotent: replace snapshot
            df.assign(date=df["date"].astype(str)).to_sql("fact_orders", con, if_exists="replace", index=False)

metrics = DataPipeline().run()
pd.Series(metrics)

In [ ]:
# 🔁 Idempotency check at pipeline level: run again → identical output
before = pd.read_csv(PROCESSED / "processed_orders.csv").shape
DataPipeline().run()
after = pd.read_csv(PROCESSED / "processed_orders.csv").shape
print("Shape before:", before, "| after 2nd run:", after, "→ identical ✅" if before == after else "→ DIFFERENT ❌")

## 🧪 17. Testing Data Pipelines
**Unit tests** (functions) • **Data tests** (nulls, uniqueness, ranges) • **Integration tests** (API → pipeline → DB).

In [ ]:
# Unit tests (run with pytest in a real project: tests/test_pipeline.py)
def calculate_total(price, quantity):
    return price * quantity

def test_calculate_total():
    assert calculate_total(100, 2) == 200

def test_clean_price_handles_commas():
    out = clean_data({"customers": clean["customers"], "orders": orders_raw,
                      "products": pd.DataFrame({"product_id": [1], "name": ["X"], "price": ["1,200"]})})
    assert out["products"]["price"].iloc[0] == 1200

def test_validation_rejects_bad_quantity():
    o = pd.DataFrame({"order_id": [1], "customer_id": [1.0], "product_id": [101], "quantity": [0], "date": [pd.Timestamp("2024-02-01")]})
    valid, rejected = validate_orders(o, clean["customers"], clean["products"])
    assert len(valid) == 0 and rejected["reject_reason"].iloc[0] == "invalid_quantity"

def test_pipeline_output_has_no_duplicate_orders():
    assert pd.read_csv(PROCESSED / "processed_orders.csv")["order_id"].is_unique

for t in (test_calculate_total, test_clean_price_handles_commas, test_validation_rejects_bad_quantity, test_pipeline_output_has_no_duplicate_orders):
    t(); print("✅", t.__name__)

## 🤖 18. ML Data Pipeline, 🧠 Training, ⚡ Inference & 🔥 Feature Pipeline

```text
TRAINING:  Collect → Validate → Clean → Feature Engineering → Split → Train → Evaluate → Register
INFERENCE: New data → Validate → Preprocess → Features → Model → Prediction → Response
```
An ML pipeline **contains** data-pipeline steps + model-specific steps (train, evaluate, deploy).

**Feature pipeline example:** Orders → aggregate → `total_orders`, `total_spending`, `avg_order_value` → ML features.

In [ ]:
# 🔥 Feature pipeline: build customer features from data BEFORE a cutoff date; label = ordered AFTER the cutoff
# (uses only past data for features → avoids data leakage)
df = pd.read_csv(PROCESSED / "processed_orders.csv", parse_dates=["date"])
cutoff = df["date"].quantile(0.6)

past, future = df[df["date"] <= cutoff], df[df["date"] > cutoff]

features = past.groupby("customer_id").agg(
    total_orders=("order_id", "count"),
    total_spending=("total_amount", "sum"),
    avg_order_value=("total_amount", "mean"),
    days_since_last=("date", lambda s: (cutoff - s.max()).days),
).reset_index()

features["will_order_again"] = features["customer_id"].isin(future["customer_id"]).astype(int)
print("Cutoff:", cutoff.date(), "| customers:", len(features), "| positive rate:", features["will_order_again"].mean().round(2))
features.head()

In [ ]:
# 🧠 Training pipeline
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

FEATURES = ["total_orders", "total_spending", "avg_order_value", "days_since_last"]
X, y = features[FEATURES], features["will_order_again"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

model = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])
model.fit(X_tr, y_tr)                                        # scaler learns from TRAIN only
print("Test accuracy:", round(accuracy_score(y_te, model.predict(X_te)), 3), "(tiny demo dataset — don't over-read!)")

In [ ]:
# ⚡ Inference pipeline: validate → preprocess → predict
def predict_customer(record: dict) -> dict:
    missing = set(FEATURES) - set(record)
    if missing:
        raise ValueError(f"Missing features: {sorted(missing)}")
    if record["total_orders"] < 0 or record["total_spending"] < 0:
        raise ValueError("Negative values are invalid")
    row = pd.DataFrame([record])[FEATURES]
    proba = float(model.predict_proba(row)[0, 1])
    return {"will_order_again": proba >= 0.5, "probability": round(proba, 3)}

print(predict_customer({"total_orders": 5, "total_spending": 150000, "avg_order_value": 30000, "days_since_last": 2}))

## 🤖 19. RAG Document-Ingestion Pipeline 🔥 (Your Future AI Engineering)

```text
PDF / Website / Doc → Extract text → Clean → Chunk → Metadata → Embeddings → Vector DB → Retrieval
```
When a document changes → re-process → **update** the vector DB (idempotency again!).

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import normalize

documents = {
    "docker_notes.txt": "Docker packages an application with its dependencies into a container so it runs the same everywhere. "
                        "A Dockerfile describes the image. Containers are lightweight compared with virtual machines.",
    "airflow_notes.txt": "Apache Airflow orchestrates pipelines using DAGs. Tasks have dependencies, schedules, retries and logs. "
                         "Airflow lets you monitor failed tasks and rerun them from the UI.",
    "stats_notes.txt": "The median is resistant to outliers while the mean is not. Standard deviation measures spread. "
                       "A p-value is not the probability that the null hypothesis is true.",
}

def chunk_text(text, size=18, overlap=4):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

# extract → clean → chunk → metadata
records = []
for doc_id, text in documents.items():
    for i, chunk in enumerate(chunk_text(" ".join(text.split()))):      # collapse whitespace = cleaning
        records.append({"chunk_id": f"{doc_id}#{i}", "source": doc_id, "text": chunk})
chunks = pd.DataFrame(records)

# embeddings (hashing vectorizer = simple stand-in for an embedding model) → in-memory "vector DB"
vec = HashingVectorizer(n_features=512, alternate_sign=False, stop_words="english")
vectors = normalize(vec.transform(chunks["text"]))

def retrieve(query, k=2):
    sims = (vectors @ normalize(vec.transform([query])).T).toarray().ravel()   # cosine similarity
    top = sims.argsort()[::-1][:k]
    return chunks.iloc[top].assign(score=sims[top].round(3))[["chunk_id", "score", "text"]]

print(f"{len(chunks)} chunks indexed")
retrieve("how does airflow rerun failed tasks?")

## 🐳 20. Docker, ☁️ Cloud & 🧰 Tools
Docker packages the pipeline + dependencies + environment so it runs identically everywhere.

In [ ]:
# Write a Dockerfile + requirements for the pipeline (reference — build with: docker build -t ecommerce-pipeline .)
dockerfile = """FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY src/ ./src/
COPY data/ ./data/
ENV PYTHONUNBUFFERED=1
CMD ["python", "-m", "src.pipeline"]
"""
(BASE / "Dockerfile").write_text(dockerfile)
(BASE / "requirements.txt").write_text("pandas\nnumpy\nscikit-learn\n")
print(dockerfile)
print("Run:  docker build -t ecommerce-pipeline .   &&   docker run --rm -e DB_PASSWORD=$DB_PASSWORD ecommerce-pipeline")

| Tool | Main Role |
|---|---|
| Pandas | Data processing |
| SQL | Extraction / transformation |
| Python | Pipeline logic |
| **Airflow / Prefect / Dagster** | Orchestration |
| **Kafka** | Event streaming |
| **Spark** | Large-scale processing |
| **dbt** | SQL-based transformations |
| Docker | Containerization |
| AWS S3 | Object storage |
| BigQuery / Snowflake | Cloud warehouse / platform |

☁️ **Cloud architecture:** `Application → API/Events → Cloud Storage (S3) → Processing → Warehouse → Analytics/ML`

## 📈 21. Scalability & Reproducibility

- **Scalability:** works for 1,000 rows ≠ works for 100 million → memory, CPU, partitioning, parallelism, **incremental processing**, distributed systems (Spark)
- **Reproducibility:** same raw data + same pipeline version + same config → same result (essential for ML)

In [ ]:
# 📦 Process big files in chunks instead of loading everything into memory
big_total, rows_seen = 0.0, 0
for chunk in pd.read_csv(PROCESSED / "processed_orders.csv", chunksize=100):
    big_total += chunk["total_amount"].sum()
    rows_seen += len(chunk)
print(f"Processed {rows_seen} rows in chunks | total revenue = {big_total:,.0f}")

# 🎲 Reproducibility: fixed seeds + versioned config
CONFIG = {"pipeline_version": "1.0.0", "random_state": 42, "cutoff_quantile": 0.6}
print(CONFIG)

## 🧠 22. Design Principles ✅ & Common Mistakes ❌

| ✅ Do | ❌ Don't |
|---|---|
| Keep **raw** data | One huge script |
| **Validate early** | No validation |
| Make transforms reproducible | No logging |
| **Log** everything important | No retry |
| Handle failures | No monitoring |
| Make pipelines **idempotent** | Hard-coded credentials |
| Process **incrementally** | Full reprocessing every time |
| Monitor data **quality** | Ignore schema changes |

### 🎯 Design Checklist
1 Source? 2 Frequency? 3 Format? 4 Volume? 5 Batch or streaming? 6 Transformations? 7 Validation? 8 Destination? 9 Failure handling? 10 Monitoring? 11 Duplicate prevention? 12 Schema changes? 13 Security? 14 Testing? 15 Scaling?

### 🆚 Comparisons
| Data Cleaning | Data Pipeline |
|---|---|
| Improves data quality | Moves + processes + delivers data end-to-end |
| One **stage** inside a pipeline | The whole flow |

| ETL | Data Pipeline |
|---|---|
| A specific pattern (E-T-L) | Broader: ingestion, validation, cleaning, storage, monitoring, alerts |

| Workflow | Data Pipeline |
|---|---|
| Any sequence of tasks (report → email → archive) | Specifically moves & processes data |

| Data Pipeline | ML Pipeline |
|---|---|
| Extract, clean, transform, store | Data steps + feature engineering, train, evaluate, register/deploy model |

## 🧠 23. What to MEMORIZE vs REFERENCE
| 🧠 Memorize | 📖 Reference |
|---|---|
| ETL vs ELT | Airflow operator syntax |
| Batch vs streaming | Cron expressions |
| Idempotency + UPSERT | Exact SQL upsert syntax per DB |
| Full vs incremental, watermark idea | Docker commands |
| Validate early, keep raw, log, retry | Library-specific APIs |
| Bronze / Silver / Gold | Kafka / Spark configs |

## 🧪 24. Mini Project — E-Commerce Data Pipeline

You've already built the working core above (`DataPipeline` class). To turn it into a **portfolio project** 🚀:

```text
mini-project/
├── data/
│   ├── raw/{customers,products,orders}.csv
│   └── processed/{processed_orders,customer_summary,product_summary,daily_sales}.csv
├── src/{extract,clean,transform,validate,pipeline}.py
├── tests/test_pipeline.py
├── Dockerfile
└── README.md
```
**Steps:** Extract → Validate → Clean → Transform (join + `total_amount`) → Aggregate → Validate output → Load → Analyze.

💼 **Freelancing angle:** "I build automated, validated, monitored data pipelines" is a sellable Upwork/Fiverr service (clean CSV/API data → database → dashboard). Add a README with architecture diagram + screenshots.

In [ ]:
# Step 8 — Analyze the processed data
daily = pd.read_csv(PROCESSED / "daily_sales.csv", parse_dates=["date"])
prod = pd.read_csv(PROCESSED / "product_summary.csv")
cust = pd.read_csv(PROCESSED / "customer_summary.csv")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.lineplot(data=daily, x="date", y="revenue", marker="o", ax=axes[0]).set_title("Daily Revenue")
axes[0].tick_params(axis="x", rotation=45)
sns.barplot(data=prod, y="product_name", x="revenue", ax=axes[1]).set_title("Revenue by Product")
sns.histplot(cust["total_spending"], bins=15, kde=True, ax=axes[2]).set_title("Customer Spending")
plt.tight_layout(); plt.show()

print("Top 3 customers:\n", cust.nlargest(3, "total_spending")[["name", "total_orders", "total_spending"]].to_string(index=False))

## ✏️ 25. Practice Exercises

**Beginner** 1 Create a CSV dataset • 2 Load with Pandas • 3 Clean missing values • 4 Transform a column • 5 Save processed data

**Intermediate** 6 Separate `extract()`, `transform()`, `load()` functions • 7 Add validation checks • 8 Add logging • 9 Add exception handling • 10 Add duplicate detection

**Advanced** 11 Incremental processing • 12 Rejected-record file • 13 Pipeline metrics • 14 Unit tests • 15 Dockerize • 16 Scheduled pipeline • 17 ML training-dataset pipeline • 18 RAG document-ingestion pipeline

In [ ]:
# ✏️ Your practice space — Exercise 10: duplicate detection
# TODO: find duplicate rows in orders_raw by (customer_id, product_id, date, quantity) and report how many

## 🎯 26. Key Takeaways
- A Data Pipeline **extracts, moves, transforms, validates, and delivers** data for analytics, apps, or ML
- **Batch** = scheduled groups | **Streaming** = continuous | **Micro-batch** = small frequent batches
- **ETL** transforms before loading; **ELT** loads first, transforms in the destination
- **Incremental** processes only new/changed data; **CDC** detects inserts/updates/deletes
- **Idempotent** pipelines can be re-run safely (UPSERT, unique IDs, dedup)
- **Validate early**, **quarantine** bad records, **log** everything, **retry** transient failures (with limits)
- Keep **raw** data; use **Bronze → Silver → Gold**
- ⭐ *Pipeline success ≠ data quality success*

> **Data Cleaning** fixes data • **EDA** investigates • **Statistics** reasons • **Feature Engineering** creates signals • **Data Pipelines** reliably move and process it at scale

## 💼 27. Interview Questions
**Beginner:** 1 What is a Data Pipeline? 2 Why important? 3 What is ingestion? 4 Common data sources? 5 Batch processing? 6 Stream processing? 7 Batch vs streaming? 8 ETL? 9 ELT? 10 ETL vs ELT?

**Intermediate:** 11 Incremental loading? 12 Full loading? 13 CDC? 14 Data validation? 15 Data quality? 16 Data lineage? 17 DAG? 18 Orchestration? 19 Why logging? 20 Handling failures? 21 Idempotency? 22 Preventing duplicates? 23 Data warehouse? 24 Data lake? 25 Lake vs warehouse? 26 Lakehouse? 27 Schema validation? 28 Schema evolution?

**Advanced:** 29 Design a scalable pipeline? 30 Process 100M records? 31 Design a real-time pipeline? 32 Late-arriving data? 33 Duplicate events? 34 Monitor data quality? 35 Recover from failures? 36 Make a pipeline idempotent? 37 Design an ML data pipeline? 38 Design a RAG ingestion pipeline?

### 💼 Model Answers
- **Data Pipeline:** A sequence of processes that extracts, moves, transforms, validates, and delivers data from sources to destinations for analytics, applications, or ML.
- **ETL vs ELT:** ETL transforms before loading; ELT loads first and transforms inside the destination platform.
- **Batch vs streaming:** Batch handles data in groups at intervals; streaming processes events continuously with low latency.
- **Idempotency:** Re-running the same operation yields the same intended final state — no unintended duplicates or side effects.
- **Validation:** Ensures data meets expected quality, schema, and business rules before reaching downstream systems.
- **Incremental loading:** Processes only new or changed data instead of the entire dataset.
- **Late-arriving data (tip):** Use event-time vs processing-time, watermarks with a grace window, and reprocess/upsert affected partitions.
- **Duplicate events (tip):** Unique event IDs + dedup window + idempotent sinks (UPSERT/MERGE).

## 🗺️ Roadmap
```text
01 NumPy → 02 Pandas → 03 Matplotlib → 04 Seaborn → 05 Data Cleaning → 06 EDA
→ 07 Feature Engineering → 08 Statistics → 09 Data Pipeline → 10 Machine Learning
→ Deep Learning → GenAI / LLMs → Agentic AI → MLOps → AI Engineering → 🚀 Full-Stack AI Engineer
```